# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [8]:
library(dplyr)
library(ggplot2)
library(tidyr)

# Load Data

In [2]:
# WT NT vs. MUT NT
wtnt_vs_mutnt <- read.csv("05_DEGs/MUT_NT_vs_WT_NT_results.csv")
# Fix directionality 
wtnt_vs_mutnt$logFC <- wtnt_vs_mutnt$logFC * -1

# View
head(wtnt_vs_mutnt)

,X,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,CXCR4,3.549392,4.596446,-27.56984,5.272324e-16,8.783692e-12,26.59178
2,ZIC1,4.108480,5.119546,-22.06447,2.455659e-14,1.863298e-10,23.01222
3,AHNAK,3.484344,5.287682,-21.36344,4.267204e-14,1.863298e-10,22.55041
4,IGFBPL1,2.938979,6.049482,-21.30445,4.473705e-14,1.863298e-10,22.53352
5,TGFB2,2.571280,5.346180,-20.42629,9.175546e-14,2.619652e-10,21.81282
6,CDKN1A,-4.161246,7.053518,20.39294,9.434520e-14,2.619652e-10,21.42974


In [3]:
# Guide 6 vs. NonEffector
guide6_vs_unguided <- read.csv("05_DEGs/MUT_Guide6_vs_MUT_NonEffector_results.csv")

# View
head(guide6_vs_unguided)

,X,logFC,AveExpr,t,P.Value,adj.P.Val,B
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,AQP4,4.040767,7.299996,23.91333,6.164749e-15,1.027047e-10,24.26125
2,DUSP4,-2.114951,6.486410,-21.58206,3.585324e-14,2.779088e-10,22.66052
3,ITGA3,-2.072613,5.056516,-21.16515,5.004360e-14,2.779088e-10,22.22082
4,MLC1,1.869207,8.093071,20.46343,8.895863e-14,3.705127e-10,21.82155
5,IGFBP4,1.689975,6.298619,19.96119,1.358026e-13,3.863668e-10,21.35501
6,SELENOP,1.929662,6.423455,19.90527,1.424387e-13,3.863668e-10,21.30296


# Overlap DEG Lists

In [5]:
# Filter significant genes (adj.P.Val < 0.01)
sig_wtnt <- wtnt_vs_mutnt[wtnt_vs_mutnt$adj.P.Val < 0.01, ]
sig_guide6 <- guide6_vs_unguided[guide6_vs_unguided$adj.P.Val < 0.01, ]

# Get gene names from significant genes
genes_wtnt_sig <- sig_wtnt$X
genes_guide6_sig <- sig_guide6$X

# Find overlapping significant genes
overlapping_sig_genes <- intersect(genes_wtnt_sig, genes_guide6_sig)

# View
head(overlapping_sig_genes)

# Count
print(paste("Number of overlapping significant DEGs:", length(overlapping_sig_genes)))

[1] "AHNAK"  "TGFB2"  "CDKN1A" "COL6A2" "RIMBP2" "SCN4B"

[1] "Number of overlapping significant DEGs: 1756"


# Identify Rescue Genes

In [9]:
# Filter to only significant overlapping genes
wtnt_vs_mutnt <- wtnt_vs_mutnt %>% filter(X %in% overlapping_sig_genes)
guide6_vs_unguided <- guide6_vs_unguided %>% filter(X %in% overlapping_sig_genes)

In [10]:
# Identify genes with opposite logFC signs
opposite_sign_genes <- wtnt_vs_mutnt %>%
  filter(sign(logFC) != sign(guide6_vs_unguided$logFC)) %>%
  pull(X)

# View 
head(opposite_sign_genes)

# Count
print(paste("Number of genes with opposite logFC signs:", length(opposite_sign_genes)))

[1] "TGFB2"   "RIMBP2"  "SCN4B"   "ST8SIA3" "MAN1C1"  "CD9"

[1] "Number of genes with opposite logFC signs: 861"


In [11]:
# Filter to only significant overlapping genes with opposite logFC signs (rescue genes)
wtnt_vs_mutnt <- wtnt_vs_mutnt %>% filter(X %in% opposite_sign_genes)
guide6_vs_unguided <- guide6_vs_unguided %>% filter(X %in% opposite_sign_genes)

In [12]:
# Merge the two datasets by gene name
combined_rescue_genes <- merge(wtnt_vs_mutnt, guide6_vs_unguided, 
                                by = "X", 
                                suffixes = c("_WTvsNT", "_Guide6vsNonEffector"))

# Write combined file
write.csv(combined_rescue_genes, "05_DEGs/rescue_genes.csv", row.names = FALSE)